# Governance & Security Tests

**Release:** v1.0.0 | **Date:** 2026-03-17

## Test Cases
| Capability | Test Case | Target |
|------------|-----------|--------|
| Dynamic Data Masking | Apply masking to SSN/email, test by role | Verify CLONE/CTAS behavior |
| Row Access Policy | Create region mapping, apply RAP | 100% filtering accuracy |
| Tag Propagation | Apply tags at schema/table/column | Verify in TAG_REFERENCES |
| Access History | Execute all DML operations | 100% audit coverage |
| **Storage Comparison** | Test policies on Managed vs Customer storage | Identical enforcement |

## Storage Types Under Test
| Schema | Storage Type | Table |
|--------|--------------|-------|
| MANAGED_ICEBERG | Snowflake Managed | PATIENTS_PHI |
| TESTS | Customer Storage | PATIENTS_PHI |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE SCHEMA TESTS;
USE WAREHOUSE COMPUTE_WH;

## Test 1: Dynamic Data Masking
Apply masking policy to PII columns and verify enforcement.

In [ ]:
-- Create PATIENTS_PHI Iceberg table for HIPAA governance testing (Customer Storage)
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI (
    patient_id      INT,
    first_name      VARCHAR,
    last_name       VARCHAR,
    date_of_birth   DATE,
    ssn             VARCHAR,
    primary_phone   VARCHAR,
    city            VARCHAR,
    state           VARCHAR,
    insurance_plan  VARCHAR
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
ICEBERG_VERSION = 3;

-- Create PATIENTS_PHI on Managed Storage
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.PATIENTS_PHI (
    patient_id      INT,
    first_name      VARCHAR,
    last_name       VARCHAR,
    date_of_birth   DATE,
    ssn             VARCHAR,
    primary_phone   VARCHAR,
    city            VARCHAR,
    state           VARCHAR,
    insurance_plan  VARCHAR
)
CATALOG = 'SNOWFLAKE'
ICEBERG_VERSION = 3;

-- Insert 100K synthetic patient PHI records into Customer Storage
INSERT INTO ICEBERG_POC.TESTS.PATIENTS_PHI
SELECT
    SEQ4() + 1001                                                                                                               AS patient_id,
    ARRAY_CONSTRUCT('Maria','James','Priya','Robert','Aisha','David','Elena','Michael','Sarah','Carlos')[SEQ4() % 10]::VARCHAR   AS first_name,
    ARRAY_CONSTRUCT('Santos','OBrien','Patel','Chen','Johnson','Kim','Rodriguez','Thompson','Williams','Nguyen')[SEQ4() % 10]::VARCHAR AS last_name,
    DATEADD(day, -(SEQ4() % 36500 + 6570), '2000-01-01'::DATE)                                                                 AS date_of_birth,
    LPAD((SEQ4() % 900 + 100)::VARCHAR, 3, '0') || '-' || LPAD((SEQ4() % 90 + 10)::VARCHAR, 2, '0') || '-' || LPAD((SEQ4() % 9000 + 1000)::VARCHAR, 4, '0') AS ssn,
    '555-' || LPAD((SEQ4() % 9000 + 1000)::VARCHAR, 4, '0')                                                                    AS primary_phone,
    ARRAY_CONSTRUCT('Phoenix','Denver','Seattle','Austin','Chicago','Portland','Miami','Boston')[SEQ4() % 8]::VARCHAR            AS city,
    ARRAY_CONSTRUCT('AZ','CO','WA','TX','IL','OR','FL','MA')[SEQ4() % 8]::VARCHAR                                               AS state,
    ARRAY_CONSTRUCT('Blue Cross PPO','Aetna HMO','United Healthcare','Cigna EPO','Humana Gold','Medicare Advantage')[SEQ4() % 6]::VARCHAR AS insurance_plan
FROM TABLE(GENERATOR(ROWCOUNT => 100000));

-- Insert same data into Managed Storage
INSERT INTO ICEBERG_POC.MANAGED_ICEBERG.PATIENTS_PHI
SELECT * FROM ICEBERG_POC.TESTS.PATIENTS_PHI;

In [ ]:
-- Create HIPAA masking policies for PHI columns
CREATE OR REPLACE MASKING POLICY ICEBERG_POC.TESTS.MASK_SSN AS (val VARCHAR) RETURNS VARCHAR ->
    CASE
        WHEN IS_ROLE_IN_SESSION('ACCOUNTADMIN') THEN val
        ELSE '***-**-' || RIGHT(val, 4)
    END;

CREATE OR REPLACE MASKING POLICY ICEBERG_POC.TESTS.MASK_DOB AS (val DATE) RETURNS DATE ->
    CASE
        WHEN IS_ROLE_IN_SESSION('ACCOUNTADMIN') THEN val
        ELSE DATE_FROM_PARTS(YEAR(val), 1, 1)
    END;

CREATE OR REPLACE MASKING POLICY ICEBERG_POC.TESTS.MASK_PHONE AS (val VARCHAR) RETURNS VARCHAR ->
    CASE
        WHEN IS_ROLE_IN_SESSION('ACCOUNTADMIN') THEN val
        ELSE '555-***-****'
    END;

-- Apply masking policies to Customer Storage PATIENTS_PHI
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI MODIFY COLUMN ssn           SET MASKING POLICY ICEBERG_POC.TESTS.MASK_SSN;
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI MODIFY COLUMN date_of_birth SET MASKING POLICY ICEBERG_POC.TESTS.MASK_DOB;
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI MODIFY COLUMN primary_phone SET MASKING POLICY ICEBERG_POC.TESTS.MASK_PHONE;

-- Apply same masking policies to Managed Storage PATIENTS_PHI
ALTER ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.PATIENTS_PHI MODIFY COLUMN ssn           SET MASKING POLICY ICEBERG_POC.TESTS.MASK_SSN;
ALTER ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.PATIENTS_PHI MODIFY COLUMN date_of_birth SET MASKING POLICY ICEBERG_POC.TESTS.MASK_DOB;
ALTER ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.PATIENTS_PHI MODIFY COLUMN primary_phone SET MASKING POLICY ICEBERG_POC.TESTS.MASK_PHONE;

In [ ]:
-- Test masking enforcement on both storage types (as ACCOUNTADMIN - should see full unmasked data)
SELECT 'Customer' AS storage_type, patient_id, first_name, last_name, date_of_birth, ssn, primary_phone
FROM ICEBERG_POC.TESTS.PATIENTS_PHI LIMIT 3
UNION ALL
SELECT 'Managed', patient_id, first_name, last_name, date_of_birth, ssn, primary_phone
FROM ICEBERG_POC.MANAGED_ICEBERG.PATIENTS_PHI LIMIT 3;

-- Test CTAS behavior with masking — masked copy
CREATE OR REPLACE TABLE ICEBERG_POC.TESTS.PATIENTS_PHI_MASKED_COPY AS
SELECT * FROM ICEBERG_POC.TESTS.PATIENTS_PHI LIMIT 100;

SELECT * FROM ICEBERG_POC.TESTS.PATIENTS_PHI_MASKED_COPY LIMIT 5;

## Test 2: Row Access Policy
Create region-based filtering and verify 100% accuracy.

In [ ]:
-- Create state access mapping table (payer regional restrictions)
CREATE OR REPLACE TABLE ICEBERG_POC.TESTS.STATE_ACCESS (
    role_name     VARCHAR,
    allowed_state VARCHAR
);

INSERT INTO ICEBERG_POC.TESTS.STATE_ACCESS VALUES
    ('ANALYST_WEST', 'AZ'),
    ('ANALYST_WEST', 'CO'),
    ('ANALYST_WEST', 'WA'),
    ('ANALYST_WEST', 'OR'),
    ('ANALYST_EAST', 'FL'),
    ('ANALYST_EAST', 'MA'),
    ('ANALYST_SOUTH', 'TX'),
    ('ANALYST_SOUTH', 'GA'),
    ('ACCOUNTADMIN', 'AZ'),
    ('ACCOUNTADMIN', 'CO'),
    ('ACCOUNTADMIN', 'WA'),
    ('ACCOUNTADMIN', 'TX'),
    ('ACCOUNTADMIN', 'IL'),
    ('ACCOUNTADMIN', 'OR'),
    ('ACCOUNTADMIN', 'FL'),
    ('ACCOUNTADMIN', 'MA');

-- Create Row Access Policy (HIPAA geographic restriction)
CREATE OR REPLACE ROW ACCESS POLICY ICEBERG_POC.TESTS.STATE_RAP AS (state VARCHAR) RETURNS BOOLEAN ->
    EXISTS (
        SELECT 1 FROM ICEBERG_POC.TESTS.STATE_ACCESS
        WHERE role_name = CURRENT_ROLE() AND allowed_state = state
    );

-- Apply RAP to PATIENTS_PHI
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI ADD ROW ACCESS POLICY ICEBERG_POC.TESTS.STATE_RAP ON (state);

## Test 3: Tag Propagation
Apply tags at schema/table/column levels and verify in TAG_REFERENCES.

In [ ]:
-- Create HIPAA PHI classification tags
CREATE OR REPLACE TAG ICEBERG_POC.TESTS.PII_TYPE
    ALLOWED_VALUES 'SSN', 'DATE_OF_BIRTH', 'PHONE', 'NAME', 'ADDRESS';
CREATE OR REPLACE TAG ICEBERG_POC.TESTS.DATA_CLASSIFICATION
    ALLOWED_VALUES 'HIPAA_PHI', 'CONFIDENTIAL', 'INTERNAL', 'PUBLIC';
CREATE OR REPLACE TAG ICEBERG_POC.TESTS.COST_CENTER;

-- Apply HIPAA tags to PATIENTS_PHI table and PHI columns
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI SET TAG ICEBERG_POC.TESTS.DATA_CLASSIFICATION = 'HIPAA_PHI';
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI MODIFY COLUMN ssn           SET TAG ICEBERG_POC.TESTS.PII_TYPE = 'SSN';
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI MODIFY COLUMN date_of_birth SET TAG ICEBERG_POC.TESTS.PII_TYPE = 'DATE_OF_BIRTH';
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI MODIFY COLUMN primary_phone SET TAG ICEBERG_POC.TESTS.PII_TYPE = 'PHONE';
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI MODIFY COLUMN first_name    SET TAG ICEBERG_POC.TESTS.PII_TYPE = 'NAME';
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS_PHI MODIFY COLUMN last_name     SET TAG ICEBERG_POC.TESTS.PII_TYPE = 'NAME';

-- Verify tags
SELECT * FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES('ICEBERG_POC.TESTS.PATIENTS_PHI', 'TABLE'));
SELECT * FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('ICEBERG_POC.TESTS.PATIENTS_PHI', 'TABLE'));

## Test 4: Access History Audit
Verify 100% audit coverage for all operations in ACCESS_HISTORY.

In [ ]:
-- Execute various operations to generate HIPAA audit trail
SELECT COUNT(*) FROM ICEBERG_POC.TESTS.PATIENTS_PHI;
INSERT INTO ICEBERG_POC.TESTS.PATIENTS_PHI VALUES (99999, 'Audit', 'Test', '1990-01-01'::DATE, '000-00-0000', '555-0000', 'Phoenix', 'AZ', 'Blue Cross PPO');
UPDATE ICEBERG_POC.TESTS.PATIENTS_PHI SET first_name = 'Audit_Updated' WHERE patient_id = 99999;
DELETE FROM ICEBERG_POC.TESTS.PATIENTS_PHI WHERE patient_id = 99999;

-- Query ACCESS_HISTORY for PATIENTS_PHI operations (may take a few minutes to populate)
SELECT
    query_start_time,
    user_name,
    query_type,
    direct_objects_accessed
FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY
WHERE ARRAY_CONTAINS('ICEBERG_POC.TESTS.PATIENTS_PHI'::VARIANT,
    TRANSFORM(direct_objects_accessed, o -> o:objectName::STRING))
ORDER BY query_start_time DESC
LIMIT 20;